> **구분: 수업 핵심**

# 03. 카메라·녹화 입력과 최신 프레임 정책 — 60분

**영상 개념 10분 → 시간 간격 TODO 15분 → backpressure 10분 → 실행 모드 15분 → dry-run·정리 10분** 입니다. USB 카메라는 로컬에서 한 번 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 녹화 입력은 선택 재생용입니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

카메라가 자주 촬영해도 모델의 답은 늦게 도착할 수 있습니다. 모든 프레임을 순서대로 처리하면 현재 장면이 아닌 과거 장면의 답이 쌓입니다. 이 실습은 처리 간격과 최신 한 장 보관이 서로 다른 문제를 푼다는 것을 관찰합니다.

### 전체 과정에서 내 코드가 하는 일

```text
프레임 생성
  ↓
시간 간격 검사 TODO
  ↓
최신 대기 한 장
  ↓
모델 요청
  ↓
원문·시간·오류 저장
```

시간 경계와 최신 슬롯의 계약을 구현합니다. 이미 추론 중인 요청은 최신 프레임이 왔다고 자동 취소되지 않습니다. dry-run의 파일 생성과 실제 모델 추론을 구분합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 최신 한 장만 보관하면 답이 항상 실시간인가요?

**설명:** 아닙니다. 대기열 지연은 줄여도 진행 중인 추론 시간은 남습니다. 촬영 시각·요청 시각·응답 시각과 사용한 프레임을 연결해야 하며 카메라 FPS만으로 응답 지연을 설명할 수 없습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**2일차 VLM 연결:** 실제 질의에는 공용 Ollama가 필요합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 서버·모델 저장소 확인을 요청합니다. 준비 여부는 **개인 컨테이너 JupyterLab Terminal** 에서 확인합니다.

```bash
cd /workspace
/opt/course/bin/python 05.VLM/scripts/check_environment.py
```

기본 API는 `http://127.0.0.1:11434`, 모델은 `gemma4:e2b-it-qat`이며 `vision` capability가 필요합니다. 주소·모델을 따로 배정받았다면 Python 초기화 셀에서 `os.environ['OLLAMA_HOST']`·`os.environ['VLM_MODEL']`을 설정합니다. 다른 Terminal에서 설정한 환경변수는 이미 열린 커널에 자동 적용되지 않습니다. 입력은 `Data/images/05.VLM/{road.png,dog.jpg}`, 선택 영상은 `Data/videos/05.VLM/ETS.mp4`, 출력은 `Data/outputs/05.VLM`입니다. 파일·모델이 없으면 배포 자산 경로를 확인합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[01_vlm_basics.ipynb의 1·4절](01_vlm_basics.ipynb)은 이미지 한 장을 Base64로 넣고 VLM 응답을 읽는 구조, [02_image_questions.ipynb의 1절](02_image_questions.ipynb)은 관찰 대상·보이는 근거·`unknown`·출력 항목을 질문에 명시하는 방법입니다. 여기서도 프레임 **한 장** 과 질문으로 한 요청을 만듭니다. 30 FPS는 1초당 약 30장이라는 뜻이고, 이번 TODO는 전체 영상 처리기가 아니라 주어진 시각 목록에서 요청할 **인덱스** 를 고릅니다. 정지 이미지 dry-run은 Ollama 호출을 생략하며 입력·저장 경로를 먼저 검증합니다. 이전 노트북 출력 파일은 필요 없습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–5절의 수업 핵심 경로** → [2일차 슬라이드 67쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s062) · [배포 PDF 67쪽](../슬라이드/2일차.pdf#page=67)에서 75쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 같은 이미지의 요청·원문·검증 결과·지연을 저장합니다.
- **단원 마지막 연결 질문:** 공통 사례의 기대 label을 응답을 보기 전에 정합니다. 형식 통과와 업무 판단 정답을 따로 기록하고, 틀린 허용·불필요한 거부·판단 보류를 구분합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심**

## 완료 조건

- 영상이 시간 순서의 이미지 묶음임을 설명합니다.
- 정확한 간격 경계를 포함한 `select_due_indices` TODO를 통과합니다.
- 긴 queue 대신 최신 프레임 하나를 유지하는 이유를 설명합니다.
- `camera_vlm.py`를 먼저 `--dry-run`으로 실행하고 결과 JSONL 위치를 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '05.VLM/common').is_dir())
COURSE = ROOT / '05.VLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import IMAGES, VIDEOS, OUTPUTS, LOGS, ensure_output_dirs
from common.assets import compact_image_info, encode_image
from common.client import OllamaVLMClient, timing_summary
from common.loading import load_op
ensure_output_dirs()
assert (IMAGES / 'road.png').is_file() and (IMAGES / 'dog.jpg').is_file(), '먼저 강사에게 Data/images/05.VLM 배포를 확인하세요.'
print('Python:', sys.executable)
print('input:', IMAGES)
print('output:', OUTPUTS)


> **구분: 수업 핵심**

## 1. FPS와 VLM 처리 시간은 단위가 다릅니다

30 FPS 카메라는 약 0.033초마다 프레임을 만듭니다. VLM 한 요청이 10초라면 그동안 약 300장이 생깁니다. 300개 요청을 순서대로 쌓으면 결과는 현재가 아닌 오래전 장면을 설명합니다.

```text
camera: f0 f1 f2 f3 ... f299 f300 ...
VLM:    [-------- f0 처리 --------]
다음:                            가장 최신 frame 선택
```

이 자료는 한 번에 요청 하나만 실행하고, 공유 공간에는 최신 프레임 하나만 덮어씁니다. 처리할 수 없는 중간 프레임을 버리는 것이 의도된 동작입니다.


In [ ]:
# 구분: 수업 핵심 셀
fps = 30
request_seconds = 10
produced = fps * request_seconds
print(f'{fps} FPS에서 {request_seconds}초 동안 생기는 frame:', produced)
print('모두 queue에 넣지 않고 완료 시점의 최신 frame을 선택합니다.')


> **구분: 수업 핵심**

## 2. TODO — 최소 시간 간격 선택

[exercises/sampling.py](exercises/sampling.py)의 `select_due_indices(timestamps, min_interval)`을 완성하세요.

- 빈 입력은 `[]`입니다.
- 첫 프레임은 항상 고릅니다.
- 마지막으로 고른 시각과 차이가 `min_interval` 이상이면 고릅니다. 정확히 같은 경계는 포함합니다.
- timestamp는 감소하지 않는 유한한 숫자여야 합니다. Python에서 `bool`은 `int`의 하위 타입이므로 따로 거부합니다.
- 입력 list를 수정하지 않습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** timestamps는 시간순 숫자 모음이고 min_interval은 같은 단위의 양수입니다. 반환은 시각 목록이 아니라 **원래 입력의 인덱스 목록** 입니다. `[0,0.2,1.0,1.1,2.0]`, 간격 1.0 → `[0,2,4]`.
**작성 흐름 힌트:** 간격의 bool·유한성·양수 검사 → 시각들을 새 list로 읽어 타입·유한성·비감소 순서 검사 → 빈 목록 처리 → 인덱스 0 선택 → 마지막으로 **선택한** 시각과 차이가 `>= min_interval`일 때만 인덱스와 기준 시각 갱신.
**설명할 지점:** “직전 프레임과 비교하면 촘촘한 영상은 영원히 선택되지 않을 수 있습니다. last_time은 선택할 때만 바뀝니다.”
**필요한 도구·통과 기준:** 제공 starter에 없는 `math`를 import해 `math.isfinite`를 사용하거나 같은 검사를 직접 구현합니다. timestamps에 문자열 하나를 넣으면 숫자 모음으로 해석하지 않고 `ValueError`로 거부합니다. 같은 시각은 허용하되 양의 간격 때문에 중복 선택하지 않습니다. `[5,5,6]`, 간격 1 → `[0,2]`입니다. 잘못된 시간·간격은 `ValueError`, 정상 빈 입력은 `[]`이며 검증은 `check_sampling('exercise')`입니다. 실제 스레드·카메라·sleep 코드는 이 TODO의 작성 범위가 아닙니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
from tests.test_course import check_sampling
check_sampling('exercise')
print('sampling TODO PASS')


In [ ]:
# 구분: 수업 핵심 셀
# 참고 정답은 강사가 안내하는 시점에 파일 편집기에서 확인합니다.
solution_path = COURSE / 'solutions/sampling.py'
print('참고 정답 경로:', solution_path)
# 수강생 파일을 자동으로 덮어쓰지 않습니다.


### 정답 해설 — 입력부터 결과까지

**시간 선택:** 입력이 유한한 숫자이고 감소하지 않는 시간 순서인지 확인합니다. 첫 항목은 선택하고, 이후에는 마지막으로 선택한 시각과의 차이가 min_interval 이상일 때만 선택합니다. 직전의 모든 프레임이 아니라 마지막 선택 시각을 기준으로 삼습니다.

시각 `[0,0.4,1.0,1.4,2.0]`, 간격 1.0이면 index `[0,2,4]`입니다. 정확히 1초 차이도 `>=`로 포함합니다. 빈 입력은 `[]`, 감소하는 시각·NaN·0 이하 간격은 오류입니다.

이 함수는 저장된 시각의 선택 규칙입니다. 느린 실제 추론의 대기열을 최신 한 장으로 유지하는 worker/mailbox와는 역할이 다릅니다.

**확인 질문:** 선택하지 않은 0.4를 기준 시각으로 갱신하면 1.0 프레임을 잘못 건너뛰게 되는 이유는 무엇인가요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/sampling.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""03_camera_description.ipynb 참고 정답."""
import math


def select_due_indices(timestamps, min_interval):
    if isinstance(timestamps, (str, bytes)):
        raise ValueError("timestamps는 숫자 모음이어야 합니다.")
    # 새 목록으로 읽어 원본을 수정하지 않고 전체 입력을 먼저 검증합니다.
    values = list(timestamps)
    if isinstance(min_interval, bool) or not isinstance(min_interval, (int, float)) or not math.isfinite(min_interval) or min_interval <= 0:
        raise ValueError("min_interval은 bool이 아닌 유한한 양수여야 합니다.")
    if any(isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) for value in values):
        raise ValueError("timestamp는 bool이 아닌 유한한 숫자여야 합니다.")
    if any(right < left for left, right in zip(values, values[1:])):
        raise ValueError("timestamps는 감소하지 않는 시간 순서여야 합니다.")
    if not values:
        return []
    # 첫 프레임은 항상 선택하고 기준은 직전 입력이 아니라 직전 선택 시각입니다.
    selected = [0]
    last_time = values[0]
    for index, current in enumerate(values[1:], start=1):
        if current - last_time >= min_interval:
            selected.append(index)
            # 선택하지 않은 프레임에서는 이 기준 시각을 갱신하지 않습니다.
            last_time = current
    return selected
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
op = load_op('sampling')
timestamps = [0.0, 2.0, 9.999, 10.0, 10.1, 19.999, 20.0]
selected = op.select_due_indices(timestamps, 10.0)
print('all:', list(enumerate(timestamps)))
print('selected:', [(i, timestamps[i]) for i in selected])
assert selected == [0, 3, 6]


> **구분: 수업 핵심**

## 3. 최신 프레임 저장소

<!-- procedure:lesson-19-038d5f:start -->
![촬영은 계속하고 요청은 최신 한 장](../교재그림/절차/lesson-19.svg)

촬영(1–2)과 요청(3–6)은 동시에 진행합니다. 추론을 기다리는 동안에도 최신 슬롯은 새 프레임으로 교체됩니다.
<!-- procedure:lesson-19-038d5f:end -->


`LatestFrame.put`은 이전 값을 새 값으로 덮어씁니다. sequence 번호가 있어 처리자가 이미 본 값인지 구분합니다. Lock은 capture thread와 VLM thread가 같은 변수를 동시에 바꾸는 순간의 충돌을 막습니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.sampling import LatestFrame
latest = LatestFrame()
latest.put('frame-1', 1.0)
latest.put('frame-2', 2.0)
latest.put('frame-3', 3.0)
print('sequence 0 이후 최신 값:', latest.get_after(0))
print('sequence 3 이후 새 값:', latest.get_after(3))
assert latest.get_after(0)[2] == 'frame-3'
assert latest.get_after(3) is None


> **구분: 수업 핵심**

## 4. 공동 USB 실습과 선택 재생

| 사용자 | 입력 | GUI | 권장 첫 실행 |
|---|---|---|---|
| 공동 USB 실습 | Jetson USB camera | Jetson 모니터에서만 `--preview` | `--source camera --dry-run` |
| 선택 재생·복습 | `Data/videos/05.VLM/ETS.mp4` | 없음 | `--source video --dry-run` |
| 모두 | `Data/images/05.VLM/*` | Notebook inline | `--source images --dry-run` |

`camera_vlm.py`는 긴 작업이므로 Jupyter 셀보다 터미널에서 실행하고 `Ctrl+C`로 중단합니다. `finally`에서 장치와 창을 닫습니다.


In [ ]:
# 구분: 수업 핵심 셀
command = [
    sys.executable, str(COURSE / 'scripts/camera_vlm.py'),
    '--source', 'images', '--images', str(IMAGES / '*.*'),
    '--max-requests', '2', '--dry-run',
]
completed = subprocess.run(command, text=True, capture_output=True, check=True)
print(completed.stdout)
assert 'saved:' in completed.stdout


> **구분: 수업 핵심**

## 5. 실제 실행 명령

카메라/영상 모드에는 `cv2`가 필요합니다. 먼저 `--dry-run`으로 frame 선택·저장·종료를 확인하고 그다음 option을 빼서 실제 요청합니다.

**개인 컨테이너의 JupyterLab Terminal에서** 아래 위치를 먼저 고정합니다.

```bash
cd /workspace
# 선택 재생·복습: GUI 없는 녹화 입력
/opt/course/bin/python 05.VLM/scripts/camera_vlm.py --source video \
  --video Data/videos/05.VLM/ETS.mp4 --interval 10 --max-requests 3 --dry-run

# 로컬: Jetson 모니터와 USB 카메라
/opt/course/bin/python 05.VLM/scripts/camera_vlm.py --source camera \
  --camera-index 0 --interval 10 --max-requests 3 --preview --dry-run
```

camera를 열지 못하면 장치 번호, 컨테이너의 `/dev/video*` 전달, 다른 프로그램의 점유를 확인합니다. USB 카메라 실습에서는 두 수강생이 로컬 화면을 함께 확인합니다. 녹화 입력은 재생·복습·장애 대응에 선택합니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 간격 TODO·최신 프레임·공동 USB 실행 결과 기록

[2일차 슬라이드 67쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s062) · [배포 PDF 67쪽](../슬라이드/2일차.pdf#page=67)로 돌아가 **67–75쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습**

## 자습: interval 선택 실험

5초·10초·20초에서 처리 요청 수, 평균 지연과 장면 변화 누락을 비교하세요. 간격이 짧다고 항상 더 좋은 판단이 되지는 않습니다. 서버가 느린데 요청 빈도만 높이면 오래된 정보와 부하가 늘어납니다.


> **구분: 수업 핵심**

## 정리

영상 처리의 핵심은 모든 frame을 보존하는 것이 아니라 목적에 맞는 최신 정보를 제한된 속도로 소비하는 것입니다. 다음 노트북에서는 VLM의 자유로운 문장을 JSON 계약으로 검사해 이후 행동 코드가 예측 가능한 입력만 받게 합니다.
